# BTL Học sâu - Đề tài 8: word2vec tiếng Việt
Bật **Internet** + **GPU (T4)** trong Settings của notebook. Upload thư mục `btl_w2v_vi` thành Kaggle Dataset (ví dụ tên `btl-w2v-vi`) rồi Add Data.

In [ ]:
!cp -r /kaggle/input/btl-w2v-vi /kaggle/working/btl && cd /kaggle/working/btl && pip install -q -r requirements.txt

In [ ]:
import os
os.environ['PYTHONHASHSEED']='0'
os.environ['BTL_WORK']='/kaggle/working/work'
%cd /kaggle/working/btl

## 0. Kiểm tra nhanh đường ống bằng dữ liệu giả (vài giây)
Bỏ qua nếu không cần. Sau đó **xoá** `/kaggle/working/work` trước khi chạy thật.

In [ ]:
!BTL_WORK=/tmp/smoke python src/prepare_data.py --smoke --n_jobs 1 && BTL_WORK=/tmp/smoke python src/train_w2v.py --epochs 1

## 1. Dữ liệu (nếu tải UIT-VSFC qua HF lỗi, thêm `--vsfc_dir <thư mục>`)

In [ ]:
!python src/prepare_data.py --n_articles 30000

## 2. Huấn luyện embedding (word2vec x8, SVD x4, GloVe x4)

In [ ]:
!python src/train_w2v.py --workers 1
!python src/cooc_svd_glove.py

## 3. Đánh giá nội tại

In [ ]:
!python src/eval_intrinsic.py

## 4. Đánh giá ngoại tại (sentiment, rồi topic) + PhoBERT

In [ ]:
!python src/eval_extrinsic.py --task sentiment --stage all --neural_kv w2v_sg_w3_d300
!python src/eval_phobert.py --task sentiment --mode both

In [ ]:
!python src/eval_extrinsic.py --task topic --stage all --neural_kv w2v_sg_w3_d300
!python src/eval_phobert.py --task topic --mode both

## 5. Ablation ngữ liệu, thiên kiến, tổng hợp

In [ ]:
!python src/corpus_ablation.py --cfg sg_w3_d300
!python src/bias_analysis.py
!python src/aggregate.py --task sentiment

## 6. Đóng gói kết quả để tải về (models rất nặng: chỉ tải results + data_stats)

In [ ]:
!cd /kaggle/working/work && zip -r /kaggle/working/results.zip results data/*.jsonl